# Raw TBMM Transcript Scraper



In [ ]:
# Colab setup
# Run this once. pyarrow is optional but enables parquet output.
%pip install -q requests beautifulsoup4 pandas pyarrow


## Imports

In [ ]:
# -*- coding: utf-8 -*-
"""Raw-only scraper for official TBMM Genel Kurul tutanak transcripts.

This script collects one row per parliamentary session from TBMM
"Tutanaklar / Donemin Tutanak Metinleri" pages.

It only stores metadata and the raw transcript text.
"""

from __future__ import annotations

import json
import logging
import re
import time
from pathlib import Path
from typing import Iterable
from urllib.parse import urldefrag, urljoin, urlparse, parse_qs

import pandas as pd
import requests
from bs4 import BeautifulSoup



## Mount Google Drive


In [ ]:
# Mount Google Drive before scraping so outputs are written directly to Drive.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("google.colab is not available; skipping Drive mount outside Colab.")


Mounted at /content/drive


## Configuration

In [ ]:
# @title
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

BASE_URL = "https://www.tbmm.gov.tr"
LIST_URL_TEMPLATE = BASE_URL + "/Tutanaklar/DoneminTutanakMetinleri?Donem={donem}&YasamaYili={yasama_yili}"

REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (compatible; TBMM-raw-tutanak-scraper/1.1; "
        "academic project; respectful single-threaded requests)"
    )
}

# TBMM pages can be slow from Colab. Use separate connect/read timeouts:
# - connect timeout: fail reasonably fast if the network cannot connect
# - read timeout: wait longer while TBMM prepares/sends a large HTML page
CONNECT_TIMEOUT_SECONDS = 20
READ_TIMEOUT_SECONDS = 180
REQUEST_TIMEOUT = (CONNECT_TIMEOUT_SECONDS, READ_TIMEOUT_SECONDS)

MAX_RETRIES = 5
RETRY_BACKOFF_SECONDS = 3
SLEEP_BETWEEN_REQUESTS = 1.0
SAVE_EVERY_N_SESSIONS = 1

# Manual target list. Keep this explicit for reproducible project runs.
# Extend later, for example: [(27, 6), (27, 5), ...]
term_year_pairs = [

    # 28. Dönem
    #(28, 4),
    #(28, 3),
    #(28, 2),
    #(28, 1),

    # 27. Dönem
    #(27, 6),
    #(27, 5),
    #(27, 4),
    #(27, 3),
    #(27, 2),
    #(27, 1),

    # # 26. Dönem
    # (26, 3),
    # (26, 2),
    # (26, 1),

  #   # 25. Dönem
  #   (25, 2),
  #   (25, 1),

  #  # 24. Dönem
  #   (24, 5),
  #   (24, 4),
  #   (24, 3),
  #   (24, 2),
  #   (24, 1),

    # 23. Dönem
    (23, 5),
    (23, 4),
    (23, 3),
    (23, 2),
    (23, 1),

  #   # 22. Dönem
  #   (22, 5),
  #   (22, 4),
  #   (22, 3),
  #   (22, 2),
  #   (22, 1),
]

OUTPUT_DIR = Path("/content/drive/MyDrive/tbmm_raw_tutanaklar")
CSV_PATH = OUTPUT_DIR / "raw_tutanaklar.csv"
JSONL_PATH = OUTPUT_DIR / "raw_tutanaklar.jsonl"
PARQUET_PATH = OUTPUT_DIR / "raw_tutanaklar.parquet"

REQUIRED_COLUMNS = [
    "session_id",
    "donem",
    "yasama_yili",
    "birlesim_no",
    "date",
    "list_page_url",
    "ham_page_url",
    "ozet_page_url",
    "ham_html_url",
    "raw_text",
    "raw_text_length",
    "scrape_status",
    "error_message",
]

TR_MONTHS = {
    "ocak": "01",
    "subat": "02",
    "mart": "03",
    "nisan": "04",
    "mayis": "05",
    "haziran": "06",
    "temmuz": "07",
    "agustos": "08",
    "eylul": "09",
    "ekim": "10",
    "kasim": "11",
    "aralik": "12",
    # Common damaged-encoding fallbacks seen in copied notebooks.
    "?ubat": "02",
    "may?s": "05",
    "a?ustos": "08",
    "eyl?l": "09",
    "kas?m": "11",
    "aral?k": "12",
}

session = requests.Session()
session.headers.update(REQUEST_HEADERS)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)


## Scraper Functions

In [ ]:
# ---------------------------------------------------------------------------
# Small utilities
# ---------------------------------------------------------------------------

def make_session_id(donem: int, yasama_yili: int, birlesim_no: int) -> str:
    """Return the required session_id format."""
    return f"{donem}_{yasama_yili}_{birlesim_no}"


def absolute_url(base_url: str, href: str | None) -> str | None:
    """Convert a possibly relative href into an absolute URL and remove fragments."""
    if not href:
        return None
    return urldefrag(urljoin(base_url, href))[0]


def repair_mojibake(value: str) -> str:
    """Repair common UTF-8-as-Windows-125x mojibake.

    Examples fixed by this function:
    - May\u00c4\u00b1s -> May\u0131s
    - \u00c5\u017eubat -> \u015eubat
    - Aral\u00c4\u00b1k -> Aral\u0131k
    """
    value = value or ""
    mojibake_markers = ("\u00c3", "\u00c4", "\u00c5", "\u00c2")
    if not any(marker in value for marker in mojibake_markers):
        return value

    original_badness = sum(value.count(marker) for marker in mojibake_markers)
    for encoding in ("cp1252", "latin1"):
        try:
            fixed = value.encode(encoding).decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            continue
        fixed_badness = sum(fixed.count(marker) for marker in mojibake_markers)
        if fixed_badness < original_badness:
            return fixed
    return value


def clean_cell_text(value: str) -> str:
    """Light normalization for table cell metadata."""
    value = repair_mojibake((value or "").replace("\xa0", " "))
    return re.sub(r"\s+", " ", value).strip()


def normalize_turkish_month_name(value: str) -> str:
    """Normalize Turkish month names to ASCII keys used in TR_MONTHS."""
    value = repair_mojibake(clean_cell_text(value)).casefold()
    translation = str.maketrans({
        "\u00e7": "c",
        "\u011f": "g",
        "\u0131": "i",
        "\u00f6": "o",
        "\u015f": "s",
        "\u00fc": "u",
    })
    return value.translate(translation).replace("i\u0307", "i")


def parse_turkish_date(value: str) -> str | None:
    """Convert dates such as '20 Mayis 2026 Carsamba' to '2026-05-20'."""
    value = clean_cell_text(value)
    match = re.search(r"(\d{1,2})\s+([^\s\d]+)\s+(\d{4})", value)
    if not match:
        return None

    day = match.group(1).zfill(2)
    month_token = match.group(2)
    month_key = normalize_turkish_month_name(month_token)
    month = TR_MONTHS.get(month_key) or TR_MONTHS.get(month_token.casefold())
    year = match.group(3)
    if not month:
        logging.warning("Could not parse Turkish month %r from date text %r", month_token, value)
        return None
    return f"{year}-{month}-{day}"


def decode_response_text(response: requests.Response) -> str:
    """Decode TBMM pages, preferring UTF-8 and repairing common mojibake."""
    content = response.content
    content_type = response.headers.get("Content-Type", "")

    candidates: list[str] = []
    charset_match = re.search(r"charset=([^;]+)", content_type, flags=re.IGNORECASE)
    if not charset_match:
        head = content[:4096].decode("ascii", errors="ignore")
        charset_match = re.search(r"charset=[\"']?([A-Za-z0-9_\-]+)", head, flags=re.IGNORECASE)
    if charset_match:
        candidates.append(charset_match.group(1).strip().strip("\"'"))

    # TBMM list pages are usually UTF-8. Try it before requests' guessed
    # apparent_encoding, which can misclassify Turkish UTF-8 as Windows-1254.
    candidates.extend(["utf-8", "utf8", "windows-1254", "cp1254", "iso-8859-9"])

    if response.encoding:
        candidates.append(response.encoding)
    if response.apparent_encoding:
        candidates.append(response.apparent_encoding)

    seen = set()
    for enc in candidates:
        enc = (enc or "").lower()
        if not enc or enc in seen:
            continue
        seen.add(enc)
        try:
            return repair_mojibake(content.decode(enc))
        except UnicodeDecodeError:
            continue
        except LookupError:
            continue

    return repair_mojibake(content.decode("utf-8", errors="replace"))

def fetch_url(url: str) -> requests.Response:
    """Fetch a URL with timeout, retries, backoff, and polite sleeping.

    Colab can intermittently read-timeout on TBMM list pages. A requests timeout
    tuple lets us keep connection failures quick while allowing slow TBMM pages
    more time to send the response body.
    """
    last_error: Exception | None = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(url, timeout=REQUEST_TIMEOUT)
            if response.status_code in {429, 500, 502, 503, 504}:
                raise requests.HTTPError(f"retryable HTTP {response.status_code}", response=response)
            response.raise_for_status()
            time.sleep(SLEEP_BETWEEN_REQUESTS)
            return response

        except requests.exceptions.ReadTimeout as exc:
            last_error = exc
            wait_seconds = RETRY_BACKOFF_SECONDS * attempt
            logging.warning(
                "Read timeout on attempt %s/%s; waiting %ss before retry: %s",
                attempt,
                MAX_RETRIES,
                wait_seconds,
                url,
            )
            if attempt < MAX_RETRIES:
                time.sleep(wait_seconds)

        except requests.exceptions.RequestException as exc:
            last_error = exc
            wait_seconds = RETRY_BACKOFF_SECONDS * attempt
            logging.warning(
                "Fetch failed on attempt %s/%s; waiting %ss before retry: %s | %s",
                attempt,
                MAX_RETRIES,
                wait_seconds,
                url,
                exc,
            )
            if attempt < MAX_RETRIES:
                time.sleep(wait_seconds)

    raise RuntimeError(f"Could not fetch {url}: {last_error}")



# ---------------------------------------------------------------------------
# Required scraper functions
# ---------------------------------------------------------------------------

def get_soup(url: str) -> BeautifulSoup:
    """Download a page and return a BeautifulSoup object."""
    response = fetch_url(url)
    html = decode_response_text(response)
    return BeautifulSoup(html, "html.parser")


def extract_session_links(donem: int, yasama_yili: int) -> list[dict]:
    """Extract one metadata row per Birleşim from a TBMM list page.

    The returned rows include the intermediate Ham page URL when available.
    Some TBMM rows expose the transcript through the clickable "Birleşim"
    link rather than a separate "Ham" link, so that link is used as fallback.
    """
    list_page_url = LIST_URL_TEMPLATE.format(donem=donem, yasama_yili=yasama_yili)
    soup = get_soup(list_page_url)

    table = soup.select_one("#mainTable table") or soup.find("table")
    if table is None:
        logging.warning("No table found on list page: %s", list_page_url)
        return []

    rows: list[dict] = []

    for tr in table.find_all("tr"):
        cells = tr.find_all("td")
        if not cells:
            continue

        session_anchor = None
        for anchor in tr.find_all("a", href=True):
            if "Birleşim" in clean_cell_text(anchor.get_text(" ", strip=True)):
                session_anchor = anchor
                break
        if session_anchor is None:
            continue

        session_label = clean_cell_text(session_anchor.get_text(" ", strip=True))
        number_match = re.search(r"(\d+)", session_label)
        if not number_match:
            continue
        birlesim_no = int(number_match.group(1))

        date_text = clean_cell_text(cells[1].get_text(" ", strip=True)) if len(cells) > 1 else ""
        parsed_date = parse_turkish_date(date_text)

        ham_anchor = None
        ozet_anchor = None
        for anchor in tr.find_all("a", href=True):
            link_text = clean_cell_text(anchor.get_text(" ", strip=True))
            title_text = clean_cell_text(anchor.get("title", ""))
            combined = f"{link_text} {title_text}"
            if re.search(r"\bHam\b", combined, flags=re.IGNORECASE):
                ham_anchor = anchor
            if "Özet" in combined or "Ozet" in combined:
                ozet_anchor = anchor

        # Fallback: TBMM often requires clicking "96. Birleşim" to reach Ham details.
        ham_page_url = absolute_url(list_page_url, ham_anchor["href"]) if ham_anchor else absolute_url(list_page_url, session_anchor["href"])
        ozet_page_url = absolute_url(list_page_url, ozet_anchor["href"]) if ozet_anchor else None

        rows.append(
            {
                "session_id": make_session_id(donem, yasama_yili, birlesim_no),
                "donem": donem,
                "yasama_yili": yasama_yili,
                "birlesim_no": birlesim_no,
                "date": parsed_date,
                "list_page_url": list_page_url,
                "ham_page_url": ham_page_url,
                "ozet_page_url": ozet_page_url,
                "ham_html_url": None,
                "raw_text": None,
                "raw_text_length": 0,
                "scrape_status": "listed",
                "error_message": None,
            }
        )

    return rows


def find_cdn_html_url(page_url: str | None, kind: str = "Ham") -> str | None:
    """Find the final CDN HTML transcript URL from an intermediate TBMM page.

    If page_url is already an HTML URL, it is returned directly.
    """
    if not page_url:
        return None

    normalized_page_url = page_url.lower()
    if normalized_page_url.endswith(".html"):
        return page_url

    soup = get_soup(page_url)
    kind_lower = kind.lower()

    html_links: list[str] = []
    kind_matched_links: list[str] = []

    for anchor in soup.find_all("a", href=True):
        href = absolute_url(page_url, anchor["href"])
        if not href:
            continue

        href_lower = href.lower()
        if not href_lower.endswith(".html"):
            continue
        if "cdn.tbmm.gov.tr" not in href_lower and "/tbmmweb/tutanak/" not in href_lower:
            continue

        html_links.append(href)

        anchor_text = clean_cell_text(anchor.get_text(" ", strip=True)).lower()
        title_text = clean_cell_text(anchor.get("title", "")).lower()
        if f"/{kind_lower}/" in href_lower or kind_lower in anchor_text or kind_lower in title_text:
            kind_matched_links.append(href)

    if kind_matched_links:
        return kind_matched_links[0]
    if html_links:
        return html_links[0]
    return None


def extract_raw_text_from_html(html_url: str) -> str:
    """Extract readable raw text from final CDN transcript HTML.

    This keeps speaker names, chair statements, applause notes, microphone
    notes, and procedural text. Whitespace is normalized only enough to make
    the transcript readable as plain text.
    """
    response = fetch_url(html_url)
    html = decode_response_text(response)
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style"]):
        tag.decompose()
    for br in soup.find_all("br"):
        br.replace_with("\n")

    paragraphs: list[str] = []
    for p in soup.find_all("p"):
        text = p.get_text("", strip=False)
        text = text.replace("\xa0", " ")
        text = re.sub(r"[ \t\f\v]+", " ", text)
        text = text.strip()
        if text:
            paragraphs.append(text)

    if paragraphs:
        raw_text = "\n\n".join(paragraphs)
    else:
        raw_text = soup.get_text("\n", strip=False)

    raw_text = raw_text.replace("\r\n", "\n").replace("\r", "\n")
    raw_text = raw_text.replace("\xa0", " ")
    raw_text = re.sub(r"[ \t\f\v]+", " ", raw_text)

    # Keep paragraph breaks, but avoid huge runs of blank lines.
    lines = [line.strip() for line in raw_text.split("\n")]
    normalized_lines: list[str] = []
    blank_count = 0
    for line in lines:
        if line:
            normalized_lines.append(line)
            blank_count = 0
        else:
            blank_count += 1
            if blank_count <= 1:
                normalized_lines.append("")

    return "\n".join(normalized_lines).strip()


def load_existing_output(path: str | Path) -> pd.DataFrame:
    """Load prior output for incremental scraping; return empty frame if absent."""
    path = Path(path)
    if path.exists():
        df = pd.read_csv(path, dtype={"session_id": str})
    elif path.with_suffix(".jsonl").exists():
        df = pd.read_json(path.with_suffix(".jsonl"), lines=True, dtype={"session_id": str})
    else:
        df = pd.DataFrame(columns=REQUIRED_COLUMNS)

    for column in REQUIRED_COLUMNS:
        if column not in df.columns:
            df[column] = None
    return df[REQUIRED_COLUMNS]


def save_outputs(df: pd.DataFrame) -> None:
    """Save CSV and JSONL, and Parquet when a parquet engine is installed."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df = df.copy()
    for column in REQUIRED_COLUMNS:
        if column not in df.columns:
            df[column] = None
    df = df[REQUIRED_COLUMNS]

    df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
    df.to_json(JSONL_PATH, orient="records", lines=True, force_ascii=False)

    try:
        df.to_parquet(PARQUET_PATH, index=False)
        logging.info("Saved parquet: %s", PARQUET_PATH)
    except Exception as exc:
        logging.warning("Parquet was not saved. Install pyarrow or fastparquet if needed. Error: %s", exc)

    logging.info("Saved CSV: %s", CSV_PATH)
    logging.info("Saved JSONL: %s", JSONL_PATH)


def backfill_missing_dates_from_list_pages() -> pd.DataFrame:
    """Fill missing date values in the existing Drive CSV from TBMM list pages.

    This does not re-download transcript HTML. It only revisits the list pages,
    builds a session_id -> date lookup, updates missing date cells, and saves.
    """
    df = load_existing_output(CSV_PATH)
    if df.empty:
        print("No existing output file found; nothing to backfill.")
        return df

    date_lookup: dict[str, str | None] = {}
    for donem, yasama_yili in term_year_pairs:
        try:
            for row in extract_session_links(donem, yasama_yili):
                date_lookup[row["session_id"]] = row.get("date")
        except Exception:
            logging.exception("Could not list dates for donem=%s yasama_yili=%s", donem, yasama_yili)

    missing_mask = (
        df["date"].isna()
        | df["date"].astype(str).str.strip().isin(["", "None", "nan", "NaN"])
    )
    before = int(missing_mask.sum())

    mapped_dates = df.loc[missing_mask, "session_id"].map(date_lookup)
    fillable_mask = missing_mask.copy()
    fillable_mask.loc[missing_mask] = mapped_dates.notna().values
    df.loc[fillable_mask, "date"] = mapped_dates[mapped_dates.notna()].values

    after = int((
        df["date"].isna()
        | df["date"].astype(str).str.strip().isin(["", "None", "nan", "NaN"])
    ).sum())

    save_outputs(df)
    print(f"Backfilled dates: {before - after}; missing dates remaining: {after}")
    return df


def scrape_terms(term_year_pairs: Iterable[tuple[int, int]], test_limit: int | None = None) -> pd.DataFrame:
    """Scrape selected term/year pairs into raw output files.

    Incremental behavior:
    - Existing session_id values in raw_tutanaklar.csv are skipped.
    - Partial outputs are saved after every SAVE_EVERY_N_SESSIONS new rows.
    - Errors are logged into rows and do not stop the whole scrape.

    Set test_limit=3 for a small test scrape.

    Important:
    - test_limit limits the number of NEW sessions scraped, not the number of term/year pages.
    - The returned DataFrame contains only the newly scraped rows in test mode.
    """
    existing_df = load_existing_output(CSV_PATH)
    existing_ids = set(existing_df["session_id"].dropna().astype(str)) if not existing_df.empty else set()

    all_records = existing_df.to_dict("records") if not existing_df.empty else []
    new_records = []

    new_records_since_save = 0

    for donem, yasama_yili in term_year_pairs:
        logging.info("Listing term/year: donem=%s yasama_yili=%s", donem, yasama_yili)

        try:
            listed_sessions = extract_session_links(donem, yasama_yili)
        except Exception:
            logging.exception("Could not list donem=%s yasama_yili=%s", donem, yasama_yili)
            continue

        # Most recent first. Use reverse=False if you prefer oldest-to-newest.
        listed_sessions = sorted(
            listed_sessions,
            key=lambda row: int(row["birlesim_no"]),
            reverse=True
        )

        for record in listed_sessions:
            session_id = str(record["session_id"])

            if session_id in existing_ids:
                logging.info("Skipping existing session_id=%s", session_id)
                continue

            # Stop BEFORE scraping if we already reached the test limit.
            if test_limit is not None and len(new_records) >= test_limit:
                final_df = pd.DataFrame(all_records).drop_duplicates("session_id", keep="last")
                save_outputs(final_df)
                return pd.DataFrame(new_records)[REQUIRED_COLUMNS]

            logging.info("Scraping session_id=%s", session_id)

            try:
                if not record.get("ham_page_url"):
                    record["scrape_status"] = "missing_ham_link"
                    record["error_message"] = "No Ham or Birleşim URL found on list row."
                    record["raw_text"] = None
                    record["raw_text_length"] = 0
                    record["ham_html_url"] = record.get("ham_html_url") or None

                else:
                    record["ham_html_url"] = find_cdn_html_url(record["ham_page_url"], kind="Ham")



                    if not record.get("ham_html_url"):
                        record["scrape_status"] = "missing_cdn_html"
                        record["error_message"] = "Ham page exists, but no final CDN HTML link was found."
                        record["raw_text"] = None
                        record["raw_text_length"] = 0
                    else:
                        raw_text = extract_raw_text_from_html(record["ham_html_url"])
                        record["raw_text"] = raw_text
                        record["raw_text_length"] = len(raw_text)
                        record["scrape_status"] = "ok"
                        record["error_message"] = None

            except Exception as exc:
                logging.exception("Error while scraping session_id=%s", session_id)
                record["scrape_status"] = "error"
                record["error_message"] = str(exc)[:1000]
                record["raw_text"] = record.get("raw_text") or None
                record["raw_text_length"] = len(record["raw_text"] or "")
                record["ham_html_url"] = record.get("ham_html_url") or None

            all_records.append(record)
            new_records.append(record)
            existing_ids.add(session_id)
            new_records_since_save += 1

            if new_records_since_save >= SAVE_EVERY_N_SESSIONS:
                final_df = pd.DataFrame(all_records).drop_duplicates("session_id", keep="last")
                save_outputs(final_df)
                new_records_since_save = 0

    final_df = pd.DataFrame(all_records).drop_duplicates("session_id", keep="last")
    save_outputs(final_df)

    if new_records:
        return pd.DataFrame(new_records)[REQUIRED_COLUMNS]

    return pd.DataFrame(columns=REQUIRED_COLUMNS)


## Test Scraping: First 3 Sessions

In [ ]:
# Test scraping mode
existing_before_test = load_existing_output(CSV_PATH)

if len(existing_before_test):
    print("Existing rows in Drive CSV:", len(existing_before_test))

    print("\nLatest existing sessions by donem/yasama_yili/birlesim_no:")
    display(
        existing_before_test
        .sort_values(
            ["donem", "yasama_yili", "birlesim_no"],
            ascending=[False, False, False]
        )[["session_id", "donem", "yasama_yili", "birlesim_no", "date", "scrape_status"]]
        .head(15)
        .reset_index(drop=True)
    )

    print("\nCurrent counts by donem and yasama_yili:")
    display(
        existing_before_test
        .groupby(["donem", "yasama_yili"])["birlesim_no"]
        .agg(["min", "max", "count"])
        .sort_index(ascending=False)
    )

test_df = scrape_terms(term_year_pairs, test_limit=3)

print("\nNewly scraped rows:")
display(
    test_df[["session_id", "donem", "yasama_yili", "birlesim_no", "date", "scrape_status"]]
    if len(test_df)
    else test_df
)

if len(test_df):
    print(test_df["scrape_status"].value_counts(dropna=False))
else:
    print("No new missing sessions found by the test scrape.")


Existing rows in Drive CSV: 1773

Latest existing sessions by donem/yasama_yili/birlesim_no:


,session_id,donem,yasama_yili,birlesim_no,date,scrape_status
0,28_4_96,28,4,96,2026-05-20,ok
1,28_4_95,28,4,95,2026-05-14,ok
2,28_4_94,28,4,94,2026-05-13,ok
3,28_4_93,28,4,93,2026-05-12,ok
4,28_4_92,28,4,92,2026-05-07,ok
5,28_4_91,28,4,91,2026-05-06,ok
6,28_4_90,28,4,90,2026-05-05,ok
7,28_4_89,28,4,89,2026-04-30,ok
8,28_4_88,28,4,88,2026-04-29,ok
9,28_4_87,28,4,87,2026-04-28,ok



Current counts by donem and yasama_yili:


min  max  count
donem yasama_yili                 
28    4              1   96     96
      3              1  114    114
      2              1  114    114
      1              1   18     18
27    6              1   87     87
      5              1  112    112
      4              1  105    105
      3              1  118    118
      2              1  105    105
      1              1   10     10
26    3              1  100    100
      2              1  121    121
      1              1  128    128
25    2              1    1      1
      1              1    9      9
24    5              1   92     91
      4              1  139    139
      3              1  139    139
      2              1  132    132
      1              1   11     11
23    5             85   89      5
      4            136  139      4
      3            112  115      4
      2            135  138      4
      1             11   11      1
22    5            121  121      1
      4            131  131      1
      3            127  127      1
      2            124  124      1
      1            114  114      1


Newly scraped rows:


,session_id,donem,yasama_yili,birlesim_no,date,scrape_status
0,23_5_84,23,5,84,2011-03-30,ok
1,23_5_83,23,5,83,2011-03-29,ok
2,23_5_82,23,5,82,2011-03-24,ok


scrape_status
ok    3
Name: count, dtype: int64


## Full Scraping

In [ ]:
# Full scraping mode
# Run this cell only when the 3-session test looks good.
# Incremental scraping is enabled: existing session_id rows are skipped.

full_df = scrape_terms(term_year_pairs)
display(full_df.head())
print(full_df["scrape_status"].value_counts(dropna=False))


,session_id,donem,yasama_yili,birlesim_no,date,list_page_url,ham_page_url,ozet_page_url,ham_html_url,raw_text,raw_text_length,scrape_status,error_message
0,23_5_81,23,5,81,2011-03-23,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,DÖNEM: 23 CİLT: 97 YASAMA YILI: 5\n\nTÜRKİYE B...,466412,ok,None
1,23_5_80,23,5,80,2011-03-22,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,DÖNEM: 23 YASAMA\nYILI: 5\n\nTÜRKİYE BÜYÜK MİL...,493768,ok,None
2,23_5_79,23,5,79,2011-03-17,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,DÖNEM: 23 CİLT: 96 YASAMA YILI: 5\n\nTÜRKİYE B...,290168,ok,None
3,23_5_78,23,5,78,2011-03-16,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,DÖNEM: 23 CİLT: 96 YASAMA YILI: 5\n\nTÜRKİYE B...,265442,ok,None
4,23_5_77,23,5,77,2011-03-15,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,DÖNEM: 23 CİLT: 96 YASAMA YILI: 5\n\nTÜRKİYE B...,159770,ok,None


scrape_status
ok                  467
missing_cdn_html      4
Name: count, dtype: int64


## Google Drive Output Directory


In [ ]:
# Outputs are written directly to Google Drive through OUTPUT_DIR.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR)
print("CSV path:", CSV_PATH)
print("JSONL path:", JSONL_PATH)
print("Parquet path:", PARQUET_PATH)


Output directory: /content/drive/MyDrive/tbmm_raw_tutanaklar
CSV path: /content/drive/MyDrive/tbmm_raw_tutanaklar/raw_tutanaklar.csv
JSONL path: /content/drive/MyDrive/tbmm_raw_tutanaklar/raw_tutanaklar.jsonl
Parquet path: /content/drive/MyDrive/tbmm_raw_tutanaklar/raw_tutanaklar.parquet


In [ ]:
# Path sanity check. These paths are already defined in the configuration cell.
print(CSV_PATH.exists(), CSV_PATH)


True /content/drive/MyDrive/tbmm_raw_tutanaklar/raw_tutanaklar.csv


## Basic Validation

In [ ]:
# Basic validation

df = load_existing_output(CSV_PATH)

print("Rows:", len(df))
print("Columns:", list(df.columns))
print("\nStatus counts:")
print(df["scrape_status"].value_counts(dropna=False))

required_missing = [column for column in REQUIRED_COLUMNS if column not in df.columns]
assert not required_missing, f"Missing columns: {required_missing}"

if len(df):
    print("\nRaw text length summary:")
    print(df["raw_text_length"].describe())

    print("\nRows with non-ok status:")
    display(df[df["scrape_status"] != "ok"].head(20))

    print("\nSample ok rows:")
    display(df[df["scrape_status"] == "ok"][[
        "session_id",
        "donem",
        "yasama_yili",
        "birlesim_no",
        "date",
        "ham_html_url",
        "raw_text_length",
        "scrape_status",
    ]])
else:
    print("No rows yet. Run the test or full scraping cell first.")


Rows: 2247
Columns: ['session_id', 'donem', 'yasama_yili', 'birlesim_no', 'date', 'list_page_url', 'ham_page_url', 'ozet_page_url', 'ham_html_url', 'raw_text', 'raw_text_length', 'scrape_status', 'error_message']

Status counts:
scrape_status
ok                  2240
missing_cdn_html       7
Name: count, dtype: int64

Raw text length summary:
count    2.247000e+03
mean     3.321401e+05
std      1.692743e+05
min      0.000000e+00
25%      2.368125e+05
50%      3.308060e+05
75%      4.384975e+05
max      1.060098e+06
Name: raw_text_length, dtype: float64

Rows with non-ok status:


,session_id,donem,yasama_yili,birlesim_no,date,list_page_url,ham_page_url,ozet_page_url,ham_html_url,raw_text,raw_text_length,scrape_status,error_message
1366,24_5_1,24,5,1,2014-10-01,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,NaN,NaN,0,missing_cdn_html,"Ham page exists, but no final CDN HTML link wa..."
1604,24_3_33,24,3,33,2012-12-04,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,NaN,NaN,0,missing_cdn_html,"Ham page exists, but no final CDN HTML link wa..."
1606,24_3_31,24,3,31,2012-11-29,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,NaN,NaN,0,missing_cdn_html,"Ham page exists, but no final CDN HTML link wa..."
1849,23_5_8,23,5,8,2010-10-19,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,NaN,NaN,0,missing_cdn_html,"Ham page exists, but no final CDN HTML link wa..."
2181,23_2_56,23,2,56,2008-01-30,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,NaN,NaN,0,missing_cdn_html,"Ham page exists, but no final CDN HTML link wa..."
2182,23_2_55,23,2,55,2008-01-29,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,NaN,NaN,0,missing_cdn_html,"Ham page exists, but no final CDN HTML link wa..."
2183,23_2_54,23,2,54,2008-01-24,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,NaN,NaN,0,missing_cdn_html,"Ham page exists, but no final CDN HTML link wa..."



Sample ok rows:


,session_id,donem,yasama_yili,birlesim_no,date,ham_html_url,raw_text_length,scrape_status
0,28_4_96,28,4,96,2026-05-20,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/4/9...,471813,ok
1,28_4_95,28,4,95,2026-05-14,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/4/9...,414584,ok
2,28_4_94,28,4,94,2026-05-13,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/4/9...,130983,ok
3,28_3_114,28,3,114,2025-08-29,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/3/1...,267776,ok
4,28_3_113,28,3,113,2025-07-20,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/3/1...,580580,ok
...,...,...,...,...,...,...,...,...
2242,23_1_5,23,1,5,2007-08-24,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,12781,ok
2243,23_1_4,23,1,4,2007-08-20,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,27899,ok
2244,23_1_3,23,1,3,2007-08-10,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,51807,ok
2245,23_1_2,23,1,2,2007-08-09,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,26091,ok


In [ ]:
print(CSV_PATH.exists(), CSV_PATH)
df = pd.read_csv(CSV_PATH)
print(df.shape)
print(df["scrape_status"].value_counts())
print(df["session_id"].tail())


True /content/drive/MyDrive/tbmm_raw_tutanaklar/raw_tutanaklar.csv
(2247, 13)
scrape_status
ok                  2240
missing_cdn_html       7
Name: count, dtype: int64
2242    23_1_5
2243    23_1_4
2244    23_1_3
2245    23_1_2
2246    23_1_1
Name: session_id, dtype: object


In [ ]:
df.groupby(["donem", "yasama_yili"])["birlesim_no"].agg(
    min="min",
    max="max",
    count="count"
).sort_index()

min  max  count
donem yasama_yili                 
22    1            114  114      1
      2            124  124      1
      3            127  127      1
      4            131  131      1
      5            121  121      1
23    1              1   11     11
      2              1  138    138
      3              1  115    115
      4              1  139    139
      5              1   89     89
24    1              1   11     11
      2              1  132    132
      3              1  139    139
      4              1  139    139
      5              1   92     91
25    1              1    9      9
      2              1    1      1
26    1              1  128    128
      2              1  121    121
      3              1  100    100
27    1              1   10     10
      2              1  105    105
      3              1  118    118
      4              1  105    105
      5              1  112    112
      6              1   87     87
28    1              1   18     18
      2              1  114    114
      3              1  114    114
      4              1   96     96

## 28.Dönem

In [ ]:
df_28_clean = df[
    (df["donem"] == 28) &
    (df["yasama_yili"].isin([1, 2, 3, 4]))
].copy()

print(df_28_clean.shape)
print(df_28_clean.groupby("yasama_yili")["birlesim_no"].agg(["min", "max", "count"]))

(342, 13)
             min  max  count
yasama_yili                 
1              1   18     18
2              1  114    114
3              1  114    114
4              1   96     96


In [ ]:
clean_path = "/content/drive/MyDrive/tbmm_raw_tutanaklar/raw_tutanaklar_28_clean.csv"
df_28_clean.to_csv(clean_path, index=False, encoding="utf-8-sig")

print(clean_path)

/content/drive/MyDrive/tbmm_raw_tutanaklar/raw_tutanaklar_28_clean.csv


In [ ]:
import pandas as pd

clean_path = "/content/drive/MyDrive/tbmm_raw_tutanaklar/raw_tutanaklar_28_clean.csv"

df_28_clean = pd.read_csv(clean_path)

print(df_28_clean.shape)

print(
    df_28_clean.groupby("yasama_yili")["birlesim_no"]
    .agg(["min", "max", "count"])
)

print(df_28_clean["scrape_status"].value_counts())

print("Duplicate session_id:", df_28_clean["session_id"].duplicated().sum())

(342, 13)
             min  max  count
yasama_yili                 
1              1   18     18
2              1  114    114
3              1  114    114
4              1   96     96
scrape_status
ok    342
Name: count, dtype: int64
Duplicate session_id: 0


In [ ]:
df_28_clean.sample(5)

,session_id,donem,yasama_yili,birlesim_no,date,list_page_url,ham_page_url,ozet_page_url,ham_html_url,raw_text,raw_text_length,scrape_status,error_message
330,28_1_12,28,1,12,2023-07-06,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak2023012...,TÜRKİYE BÜYÜK MİLLET MECLİSİ\n\nTUTANAK DERGİS...,158688,ok,NaN
88,28_4_20,28,4,20,2025-11-20,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/4/2...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/4/2...,TÜRKİYE BÜYÜK MİLLET MECLİSİ\n\nTUTANAK DERGİS...,381705,ok,NaN
112,28_3_106,28,3,106,2025-07-08,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/3/1...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/3/1...,TÜRKİYE BÜYÜK MİLLET MECLİSİ\n\nTUTANAK DERGİS...,282475,ok,NaN
37,28_4_71,28,4,71,2026-03-12,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/4/7...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/4/7...,TÜRKİYE BÜYÜK MİLLET MECLİSİ\n\nTUTANAK DERGİS...,241470,ok,NaN
130,28_3_88,28,3,88,2025-05-15,https://www.tbmm.gov.tr/Tutanaklar/DoneminTuta...,https://www.tbmm.gov.tr/Tutanaklar/Tutanak?Id=...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/3/8...,https://cdn.tbmm.gov.tr/TbmmWeb/Tutanak/28/3/8...,TÜRKİYE BÜYÜK MİLLET MECLİSİ\n\nTUTANAK DERGİS...,311789,ok,NaN


## Export as Parquet and JSON

In [ ]:
from pathlib import Path

output_dir = Path("/content/drive/MyDrive/tbmm_raw_tutanaklar")

jsonl_path = output_dir / "raw_tutanaklar_28_clean.jsonl"
parquet_path = output_dir / "raw_tutanaklar_28_clean.parquet"

# JSONL
df_28_clean.to_json(
    jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False
)

# Parquet
df_28_clean.to_parquet(parquet_path, index=False)

print("Saved:")
print(jsonl_path)
print(parquet_path)

for p in [jsonl_path, parquet_path]:
    print(f"{p.name}: {p.stat().st_size / 1024 / 1024:.2f} MB")

Saved:
/content/drive/MyDrive/tbmm_raw_tutanaklar/raw_tutanaklar_28_clean.jsonl
/content/drive/MyDrive/tbmm_raw_tutanaklar/raw_tutanaklar_28_clean.parquet
raw_tutanaklar_28_clean.jsonl: 124.74 MB
raw_tutanaklar_28_clean.parquet: 65.08 MB
